# Project 5 | Notebook 2: Sentiment Analysis

## Overview

This notebook performs sentiment analysis on the cleaned BRICS monetary policy 
statements dataset produced in Notebook 1. The input is 
`data/brics_mpc_cleaned.csv` — 305 statements from four central banks, with 
boilerplate stripped, dates parsed, and text preprocessed.

The analysis proceeds in two layers:

**Layer 1 — Loughran-McDonald Dictionary Sentiment (Primary)**  
The Loughran-McDonald (LM) financial dictionary is the standard lexicon for 
sentiment analysis in economics and finance research. Unlike general-purpose 
dictionaries, LM was built from financial filings and correctly classifies 
domain-specific language. For each statement, we compute:
- A positive sentiment score
- A negative sentiment score
- A net sentiment score (positive minus negative, normalised by total words)
- An uncertainty score (LM has a dedicated uncertainty word list, particularly 
  meaningful for central bank communication)

**Layer 2 — FinBERT Robustness Check**  
FinBERT is a BERT-based model fine-tuned on financial text. Unlike dictionary 
methods, it reads full sentences and understands context and negation. We run 
FinBERT on a sample of statements and compare its scores against LM to assess 
robustness. Where the two methods agree, findings are more credible.

---

## Research Question

Do BRICS central banks increasingly reference trade fragmentation and currency 
divergence themes post-2022, and does sentiment around these themes differ 
across institutions?

Sentiment trends are examined both overall and around key macro events:
- 2018: US-China trade war escalation
- 2020: COVID-19 shock
- 2022: Russia-Ukraine war and sanctions shock
- 2023–2024: Continued de-dollarisation and BRICS currency discussions

---

## Input

`data/brics_mpc_cleaned.csv` — 305 rows, 10 columns:

| Column | Description |
|--------|-------------|
| `country` | Country of the central bank |
| `central_bank` | Bank identifier (RBI, CBR, SARB, PBOC) |
| `date` | Parsed datetime |
| `title` | Statement title |
| `text` | Cleaned raw text (boilerplate stripped) |
| `url` | Source URL |
| `date_approximate` | True for 46 SARB rows with year-only dates |
| `date_original` | Original raw date string |
| `text_clean` | Preprocessed text for LDA (lowercased, lemmatised, stopwords removed) |
| `token_count` | Token count of cleaned text |

---

## Output

A sentiment-enriched dataset saved to `data/brics_mpc_sentiment.csv`, with the 
following new columns:

| Column | Description |
|--------|-------------|
| `lm_positive` | LM positive word count normalised by total words |
| `lm_negative` | LM negative word count normalised by total words |
| `lm_uncertainty` | LM uncertainty word count normalised by total words |
| `lm_net` | Net sentiment score (positive minus negative) |
| `finbert_label` | FinBERT predicted label (positive/negative/neutral) |
| `finbert_score` | FinBERT confidence score for predicted label |

---

## Notebook Structure

**Step 1 — Load data and investigate RBI date issue**  
The RBI date range showed an unexpected minimum of 2025-04-09 in Notebook 1, 
suggesting possible date parsing errors. This will be diagnosed and resolved 
before sentiment scoring begins.

**Step 2 — Download and load Loughran-McDonald dictionary**  
The LM master dictionary CSV is downloaded from the authors' website and parsed 
to extract positive, negative, and uncertainty word lists.

**Step 3 — Compute LM sentiment scores**  
Applied to the original `text` column (not `text_clean`) — the LM dictionary 
works on raw text since it relies on exact word matching against its word list.

**Step 4 — Visualise sentiment trends**  
Time series plots of net sentiment and uncertainty by bank, with key macro 
events annotated.

**Step 5 — FinBERT robustness check**  
Run FinBERT on a stratified sample of statements and compare with LM scores.

In [2]:
# ─────────────────────────────────────────────
# Project 5 | Notebook 2: Sentiment Analysis
# BRICS Monetary Policy Sentiment Analysis
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display settings
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 20)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
# Load cleaned dataset
DATA_IN  = Path("data/brics_mpc_cleaned.csv")
DATA_OUT = Path("data/brics_mpc_sentiment.csv")

df = pd.read_csv(DATA_IN, parse_dates=['date'])

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDate dtype: {df['date'].dtype}")
print(f"\nRow counts by bank:")
print(df['central_bank'].value_counts())

Shape: (305, 10)

Columns: ['country', 'central_bank', 'date', 'title', 'text', 'url', 'date_approximate', 'date_original', 'text_clean', 'token_count']

Date dtype: datetime64[ns]

Row counts by bank:
central_bank
PBOC    131
RBI      66
SARB     57
CBR      51
Name: count, dtype: int64


In [4]:
# Investigate RBI date range
rbi_df = df[df['central_bank'] == 'RBI'][['date', 'title', 'date_original']].copy()
rbi_df = rbi_df.sort_values('date')

print(f"RBI date range: {rbi_df['date'].min()} to {rbi_df['date'].max()}")
print(f"\nAll RBI dates (sorted):")
print(rbi_df.to_string())

RBI date range: 2025-04-09 00:00:00 to 2026-02-06 00:00:00

All RBI dates (sorted):
         date                                    title      date_original
65 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
47 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
17 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
29 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
53 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
23 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
11 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
41 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
35 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
59 2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
5  2025-04-09      Governor’s Statement: April 9, 2025      April 9, 2025
64 2025-06-06       Governor

In [5]:
# Check for duplicates across all banks
print("Duplicate rows (all columns):", df.duplicated().sum())
print("\nDuplicate rows (by title + date + central_bank):")
print(df.duplicated(subset=['central_bank', 'date', 'title']).sum())

# Check per bank
print("\nDuplicates per bank:")
for bank in df['central_bank'].unique():
    bank_df = df[df['central_bank'] == bank]
    dups = bank_df.duplicated(subset=['date', 'title']).sum()
    print(f"  {bank}: {dups} duplicates")

Duplicate rows (all columns): 60

Duplicate rows (by title + date + central_bank):
61

Duplicates per bank:
  RBI: 60 duplicates
  CBR: 0 duplicates
  SARB: 1 duplicates
  PBOC: 0 duplicates
